# Trabajo Integrador 2 — Un RAG reproducible con documentos reales

**Materiales desarrollados por Matías Barreto, 2026**

**Tecnicatura Superior en Ciencias de Datos e Inteligencia Artificial — IFTS N.º 24**
* **Nomenclatura oficial:** Procesamiento de Lenguaje Natural
* **Nombre de trabajo:** Asistente de Consulta sobre el Régimen Académico de Educación Secundaria
* **Estudiante:** Camila Amendolia

---

## El desafío

Construí **individualmente** un sistema que responda preguntas sobre un corpus de documentos reales que vos elegiste. El sistema debe buscar evidencia **antes** de generar una respuesta, mostrar los fragmentos que recibió el modelo y permitir que otra persona lo ejecute desde tu repositorio. No se evalúa la espectacularidad de la interfaz: se evalúa que puedas explicar, comprobar y reproducir el recorrido desde el archivo hasta la respuesta.

Este cuaderno es una **guía de planificación y entrega**. Para aprender e implementar cada pieza, seguí los cuadernos de `014_rag_local`: documentos → fragmentos → embeddings y Chroma → respuesta con fuentes → Gradio → pruebas. Los documentos ficticios de Acme y de la biblioteca sirven para practicar, **no cuentan como documentos reales de tu entrega**.

**Entrega:** miércoles **18 de noviembre de 2026, a las 12:00 del mediodía**. No esperes al último momento para comprobar permisos del video y acceso al repositorio.

## Qué tenés que entregar

1. Un **repositorio GitHub** con código, `README.md`, `requirements.txt`, los documentos con los que probaste el RAG y pruebas que permitan inspeccionar resultados. No publiques `.env`, claves, datos personales ni el índice vectorial generado.
2. Una aplicación **Gradio** ejecutable localmente que permita preguntar y ver por separado respuesta y fragmentos recuperados, con archivo y página cuando corresponda.
3. Un video de demostración en **Drive**, de **cinco minutos como máximo**, donde muestres el funcionamiento y una limitación real del sistema.
4. El **formulario de entrega**, completado desde tu **correo institucional**, con cinco campos: **apellido, nombre, DNI, enlace al video en Drive y enlace al repositorio GitHub**. Comprobá que los enlaces permitan acceder a la evaluación.

El **DNI va sólo en el formulario**; no lo incluyas en GitHub, en el video ni en el README. Si el formulario todavía no está publicado, prepará los enlaces y esperá la indicación de la cátedra para enviarlos. Esta entrega es individual, aunque hayas intercambiado ideas o recibido asistencia.

## Condiciones para aprobar

| Componente obligatorio | Evidencia que debe verse |
|---|---|
| Caso y corpus | Una persona destinataria, una necesidad concreta y **al menos tres documentos reales** que puedan compartirse legalmente; los mismos archivos están en GitHub. Cada fuente tiene procedencia identificada. |
| Ingesta y fragmentación | Carga reproducible de los archivos, texto extraído inspeccionado, metadatos de fuente (y página para PDF), tamaño y solapamiento justificados. |
| Recuperación | Embeddings con `bge-m3` en Ollama (o alternativa justificada), índice Chroma y consultas que muestran los fragmentos recuperados. El modelo de consulta es el mismo que el del índice. |
| Generación | Un modelo local con Ollama —la ruta enseñada usa `granite4:latest`— responde con un contexto explícito y se puede identificar qué fragmentos recibió. |
| Fuentes y límites | La interfaz Gradio muestra respuesta **y** fragmentos de respaldo. Incluí una pregunta sin respuesta en el corpus y revisá si el sistema se abstiene; no afirmes que la abstención está garantizada. |
| Pruebas | Al menos **tres preguntas respondibles** (una por documento) y **dos sin respuesta**. Anotá en una tabla qué fuente esperabas, qué recuperó el sistema y qué hizo el generador. |
| Reproducción | `README.md` con requisitos, instalación por `uv`, descarga de modelos, forma de indexar y ejecutar Gradio, corpus, resultados de prueba, decisiones y límites. Otra persona puede ejecutar el proyecto con los archivos publicados. |
| Transparencia sobre IA | Si usaste IA para construir el proyecto, indicá en el README **para qué**, **en qué momentos** y **qué modelos** utilizaste, y qué verificaste personalmente. |

Un gráfico bonito no reemplaza la evidencia. Una cita redactada por el modelo tampoco: hay que poder volver al fragmento y al documento original. Si un PDF está escaneado, necesitás texto extraíble u OCR documentado; un archivo ilegible no cuenta como fuente indexada.

## 1. Delimitar un problema y seleccionar el corpus

Este sistema permite que el personal educativo consulten fechas de periodos de intensificación, maneras de evaluar, diferentes modelos. En Documentos de Apoyo N° 5, N° 6 y N° 10 de la Dirección Provincial de Educación Secundaria (DGCyE) para planificar e implementar los períodos. 

| Documento | Autor u organismo | Enlace o procedencia | Fecha/versión | Permiso o licencia | Pregunta respondible |
|---|---|---|---|---|---|
| 1 Documento de Apoyo N° 5  |Dirección General de Cultura y Educación (DGCyE) - Dirección Provincial de Educación Secundaria (PBA)  |https://abc.gob.ar/secretarias/areas/subsecretaria-de-educacion/educacion-secundaria/educacion-secundaria/regimen-academico |2024 | Pública| ¿Cuáles son los cuatro modelos de organización institucional para intensificar materias pendientes y en qué consiste cada uno?|
| 2 Documento de Apoyo N° 6 | Dirección General de Cultura y Educación (DGCyE) - Dirección Provincial de Educación Secundaria (PBA) | https://abc.gob.ar/secretarias/areas/subsecretaria-de-educacion/educacion-secundaria/educacion-secundaria/regimen-academico | 2024 | Pública | ¿En qué tres grupos se organizan las trayectorias educativas para planificar la enseñanza y cuántas instancias anuales tiene cada grupo? |
| 3  Documento de Apoyo N° 10| Dirección General de Cultura y Educación (DGCyE) - Dirección Provincial de Educación Secundaria (PBA) | https://abc.gob.ar/secretarias/areas/subsecretaria-de-educacion/educacion-secundaria/educacion-secundaria/regimen-academico | 2025 | Pública| ¿A qué estudiantes está dirigida la intensificación en el período de julio y a quiénes en el período de agosto respecto de las materias del año en curso? |

## 2. Hacer visible el pipeline

```text
documentos reales (versionados en GitHub)
    → carga y revisión del texto y los metadatos
    → fragmentos con procedencia
    → embeddings locales → Chroma
pregunta → embedding del mismo modelo → fragmentos recuperados
    → contexto con fuente y página → prompt → Ollama
    → respuesta + fragmentos mostrados por Gradio
```

La ruta de clase usa `PyPDFLoader` y `TextLoader` (`langchain-community`), `RecursiveCharacterTextSplitter` (`langchain-text-splitters`), `OllamaEmbeddings` y `ChatOllama` (`langchain-ollama`), `Chroma` (`langchain-chroma`) y `ChatPromptTemplate` (`langchain-core`). **No copies imports antiguos** como `from langchain.chains import RetrievalQA`: seguí los cuadernos actualizados y comprobá que cada paso de tu versión funciona.

| Decisión | Qué tenés que explicar | Qué probar |
|---|---|---|
| `chunk_size` y `chunk_overlap` | Qué unidad miden y por qué ese corte no separa demasiado la evidencia. | Buscá una respuesta partida o incompleta y ajustá un valor. |
| Modelo de embeddings | Cuál usaste para **indexar y consultar**. | Si lo cambiás, creá otro índice: no mezcles vectores de distintos modelos. |
| `k` | Cuántos fragmentos entran al contexto. | Mirá si faltan fragmentos o entra información irrelevante. |
| Prompt | Qué se considera evidencia y qué ocurre si falta. | Probá una pregunta sin respuesta: el modelo puede fallar aunque la instrucción sea clara. |

No hace falta una arquitectura compleja ni funciones ocultas en `utils.py`. Mostrá los pasos analíticos en el notebook o en archivos de aplicación legibles; las utilidades de visualización, si las usás, sólo deben dibujar resultados.

| Decisión | Explicación técnica en el proyecto | Prueba realizada y resultado observado |
|---|---|---|
| **`chunk_size` y `chunk_overlap`** | Miden la longitud máxima del fragmento y el solapamiento entre cortes consecutivos en **caracteres** (aprox. 150–200 palabras para 1100 caracteres). Elegí `chunk_size=1100` y `chunk_overlap=180` porque los textos normativos de la DGCyE (Res. 1650/24 y Documentos de Apoyo N° 5, 6 y 10) desarrollan condiciones y excepciones en párrafos largos que no deben quedar truncados. | **Antes (`500 / 80`):** Al consultar por los períodos de intensificación de julio y agosto, el corte partía la oración normativa antes de indicar a quiénes iba dirigida.<br>**Después (`1100 / 180`):** Se indexaron 105 fragmentos completos, bajó la distancia semántica (`0.528`) y se recuperó la regla íntegra en un solo bloque. |
| **Modelo de embeddings** | Utilicé **`bge-m3`** (vía `OllamaEmbeddings`, 1024 dimensiones) tanto para **indexar** los 105 fragmentos en `chroma_db_dgcye_ra` como para **vectorizar cada consulta** del usuario, por su buen rendimiento multilingüe en español administrativo y educativo. | Se verificó que el índice persistente se reconstruye limpio (`delete_collection()`) ante cambios de configuración para garantizar que todos los vectores pertenezcan al mismo espacio vectorial de `bge-m3` sin mezclar modelos. |
| **`k` (cantidad de fragmentos)** | Define cuántos fragmentos (`k=3`) entran al contexto del LLM tras la búsqueda por similitud (`similarity_search_with_score`). Con 3 fragmentos se cubren hasta ~3.300 caracteres de evidencia sin saturar la ventana de contexto ni diluir la atención del modelo local. | Se observó que en páginas extensas (como la p. 1 del Doc. N° 10), `k=3` a veces recupera dos cortes consecutivos de la misma página (`[1]` y `[2]`). Al subir `k` o reformular preguntas, aumenta la cobertura de reglas distribuidas en varias páginas, pero también el riesgo de cruzar referentes entre fragmentos (como ocurrió entre el primer y segundo grupo en el Doc. N° 6). |
| **Prompt (`ChatPromptTemplate`)** | Define como **única evidencia válida** los bloques numerados `[1]`, `[2]` y `[3]` inyectados en el mensaje humano, obliga a citar cada afirmación y establece una **regla estricta de abstención**: responder literalmente *"No encuentro esa información en los documentos."* si el dato no figura en el contexto. | **Prueba de abstención:** Ante *"¿Cuál es el salario básico de un docente de escuela secundaria en la Provincia de Buenos Aires?"*, el modelo se abstuvo correctamente sin alucinar.<br>**Límite observado:** Ante consultas con siglas aisladas (*"¿Qué significa CCA?"*) o muy amplias, el recuperador no ubicó el fragmento y el modelo aplicó la abstención; además, en respuestas largas a veces añadió la frase de abstención al final por sesgo de la instrucción estricta. |

## 3. Dejar una aplicación ejecutable

Construí una interfaz **Gradio** con una entrada de pregunta y, como mínimo, dos salidas: **respuesta** y **fuentes recuperadas**. Las fuentes deben ser los `Document` que pasaron al contexto, con el nombre del archivo y la página cuando exista. Podés añadir la distancia como ayuda para inspeccionar el ranking, pero no la presentes como probabilidad de verdad.

`05_interfaz_gradio.ipynb` y `app.py` de la unidad 014 muestran cómo reutilizar el RAG construido: no implementes otro sistema distinto para la interfaz. Para la entrega, la ejecución **local** alcanza. Si elegís publicar la aplicación, eso es opcional y no puede reemplazar las instrucciones de ejecución local. No se requiere ningún orquestador de flujos, contenedor ni servicio pago.

Antes de grabar la demo, probá con el repositorio en una carpeta nueva: instalá con `uv`, descargá los modelos Ollama y ejecutá el mismo comando que figura en tu README. El índice generado puede ignorarse si el README explica cómo reconstruirlo a partir de los archivos de `datos/`.

## 4. Probar antes de concluir

Escribí los casos **antes** de evaluar el sistema. Usá al menos tres preguntas con respuesta (distribuidas entre los tres documentos) y dos cuyo dato no esté en el corpus.

| Documento propio | Pregunta respondible | Página o sección esperada | ¿Qué recuperó? | ¿Qué interpretó el modelo? | Límite observado |
|---|---|---|---|---|---|
| Documento de apoyo n°5- La intensificación de las materias pendientes de aprobación y acreditación en base a la estructura curricular.pdf | ¿Cuántos modelos hay?| Página 3 (cuadro resumen con los 4 modelos) y Página 4 (mención explícita a "modelos 1, 2, 3 o 4"). | No recuperó el cuadro de la página 3; recuperó el fragmento de la página 4 donde se menciona la asignación de los modelos por parte del equipo directivo. | Respondió enumerando solo hasta el modelo 3, omitiendo el modelo 4 a pesar de que figuraba al final de la oración recuperada. | Extracción + Generación: PyPDFLoader suele desestructurar los cuadros/tablas (p. 3), haciendo que pierdan relevancia semántica; además, el modelo cortó la enumeración antes del cierre de la frase en la p. 4. |
| DOCUMENTO DE APOYO N°6 Intensificación_Profundización.pdf | ¿Cuántos momentos del año tienen el primer y el segundo grupo para aprender los saberes enseñados y las materias no aprobadas? | Páginas 11–13 (Organización de los tiempos y momentos de intensificación y profundización). | Recuperó las páginas 12, 13 y 11 del Documento de Apoyo N° 6. | Identificó los grupos y cifras (5 y 6 momentos), pero invirtió los valores (asignó 5 al primer grupo y 6 al segundo, cuando en el texto original es 6 y 5). | Corte + Generación: Al dividir el texto en chunks de 1100 caracteres, ambos grupos quedaron repartidos entre fragmentos consecutivos ([1] y [2]) y el LLM local cruzó los referentes. |
|DPESEC-Documento de Apoyo Nro 10_RA.pdf |- ¿Qué significa CCA? - ¿Qué se debe consignar en el RITE del primer cuatrimestre? | Página 2 (elaboración y actualización del RITE - Primer Cuatrimestre). | Con la sigla aislada ("¿Qué significa CCA?") no recuperó el fragmento pertinente. Al reformular con contexto ("¿Qué se debe consignar en el RITE..."), recuperó exactamente la página 2. | 1) Ante "¿Qué significa CCA?", respondió "No encuentro esa información en los documentos.".2) Ante la consulta sobre el RITE, recuperó el texto donde figura "Continúa, Con Avances (CCA)" junto con AA y CSA. | Recuperación por embeddings densos: Las siglas cortas y sin contexto (CCA) tienen baja densidad semántica para bge-m3; el modelo necesita palabras clave del entorno documental (RITE, primer cuatrimestre) para ubicar el fragmento. |

Si falla, localizá el paso: extracción ilegible, corte que separó una condición, búsqueda que no encontró el fragmento o generación que interpretó mal la evidencia. Registrá al menos **una limitación concreta** y qué cambiarías; no ocultes un fallo con una pregunta elegida sólo para que la demo salga bien. Este pequeño conjunto de pruebas muestra el mecanismo, no mide precisión general del sistema.

## 5. Plan de trabajo: una semana

No son tres etapas aisladas: cada día dejá una versión pequeña que se pueda comprobar.

| Momento | Producto verificable |
|---|---|
| Día 1 | Problema acotado, tres documentos reales publicables, procedencia y preguntas iniciales. |
| Día 2 | Carga, revisión de texto extraído, metadatos y fragmentos legibles. |
| Día 3 | Embeddings, colección Chroma e inspección de resultados para preguntas propias. |
| Día 4 | Generación con contexto, respuesta con referencias y pregunta sin respuesta. |
| Día 5 | Gradio sobre el mismo pipeline y prueba de los cinco casos. |
| Día 6 | README, reconstrucción desde archivos publicados, correcciones y declaración de uso de IA. |
| Día 7 | Demo de hasta cinco minutos; verificar accesos a Drive y GitHub y completar el formulario **antes del miércoles 18 de noviembre a las 12:00**. |

Si falta tiempo, priorizá que carga, recuperación, respuesta y fuentes sean verificables. Una interfaz sencilla y confiable vale más que muchas opciones que no pudiste probar.

## 6. README que otra persona pueda seguir

Incluí, en este orden:

1. **Problema y destinatario.** Qué preguntas puede resolver el sistema y cuáles no.
2. **Corpus.** Tabla con todos los archivos publicados en `datos/`, origen, autoría, fecha y condiciones de uso.
3. **Arquitectura.** Flujo desde los archivos hasta la respuesta, con modelos y versiones usadas.
4. **Instalación local.** Python 3.12, `uv`, Ollama; desde la carpeta del proyecto: `uv venv --python 3.12`, `uv pip install -r requirements.txt`, `ollama pull granite4:latest`, `ollama pull bge-m3`. Explicá cómo seleccionar el entorno y cómo abrir la interfaz (`python app.py` si seguís el ejemplo).
5. **Reindexación.** Qué carpeta se genera y cómo crear un índice nuevo si cambian corpus, corte o modelo. No publiques `.venv/`, `.env` ni la base generada.
6. **Pruebas.** Tabla de cinco casos, con hallazgos y limitación observada.
7. **Decisiones y límites.** Fragmentación, `k`, modelo de embeddings, fuentes, PDF escaneado si corresponde y alcance de las citas.
8. **Uso de IA.** Si la usaste, consigná para qué tareas, en qué momentos, qué modelos y qué comprobaste personalmente. Si no la usaste, podés indicarlo.

La consigna exige GitHub y documentos versionados. Un enlace a tu computadora, a una carpeta privada o sólo a una app desplegada **no** permite reproducir el proyecto.

## 7. Video y formulario

La demo debe durar **como máximo cinco minutos**. Una estructura posible:

| Tiempo orientativo | Qué mostrar |
|---|---|
| 0:00–0:45 | Problema, destinatario y los tres documentos reales del repositorio. |
| 0:45–1:45 | Cómo se cargan, cortan y guardan con sus fuentes. |
| 1:45–3:15 | Consulta respondible en Gradio: respuesta y fragmento de respaldo comprobado en el original. |
| 3:15–4:15 | Pregunta sin respuesta o limitación real; qué hizo el sistema. |
| 4:15–5:00 | README: cómo instalar, indexar y ejecutar el proyecto. |

Subí el video a Drive y verificá sus permisos de lectura desde otra cuenta. En el formulario, usando tu **correo institucional**, completá **apellido**, **nombre**, **DNI**, **enlace al video en Drive** y **enlace a GitHub**. El DNI no debe aparecer en los materiales públicos. No hace falta editar el video ni preparar diapositivas: debe verse el sistema funcionando.

## Lista final de comprobación

- [ ] Mi entrega es individual y el formulario se envió antes del **18/11/2026 a las 12:00**.
- [ ] El repositorio contiene los **tres o más documentos reales** usados en las pruebas; puedo publicarlos legalmente.
- [ ] El pipeline carga archivos, conserva fuente y página, fragmenta, indexa en Chroma y consulta con el mismo modelo.
- [ ] La respuesta se genera con Ollama y la interfaz Gradio muestra los fragmentos que entraron al contexto.
- [ ] Documenté al menos tres consultas respondibles y dos sin respuesta, además de una limitación observada.
- [ ] El README permite instalar por `uv`, descargar modelos y reconstruir el índice desde el corpus versionado.
- [ ] Declaré propósito, momentos, modelos y verificación personal si usé IA.
- [ ] El video en Drive dura **cinco minutos o menos** y el enlace se puede abrir.
- [ ] El enlace a GitHub funciona y no publiqué `.env`, credenciales ni DNI.

## Glosario

| Término | Significado |
|---|---|
| **RAG** | Recuperación de documentos seguida de generación con esos fragmentos como contexto. |
| **Ingesta** | Lectura y preparación de los documentos que alimentan la búsqueda. |
| **Fragmento** | Parte de un documento que conserva su procedencia para ser recuperada. |
| **Embedding** | Vector producido por un modelo para comparar textos. |
| **Índice vectorial** | Colección de vectores y textos usada por el buscador; se reconstruye desde documentos compartidos. |
| **Fuente recuperada** | Fragmento enviado al modelo; no prueba por sí solo que la respuesta lo haya interpretado bien. |
| **Abstención** | Reconocimiento de que el corpus no alcanza para responder; se solicita y se comprueba. |
| **Reproducibilidad** | Capacidad de otra persona de ejecutar y verificar el sistema con código, documentos e instrucciones publicados. |